# ViGen AIO Studio - VieNeu-TTS Colab Server

Notebook này khởi chạy máy chủ VieNeu-TTS (Tiếng Việt truyền cảm) trên Google Colab với hỗ trợ Async Task Queue, Google Drive Auto-Sync và Ngrok Tunnel.

### Hướng dẫn sử dụng:
1. **Thiết lập GPU**: Chọn **Runtime** -> **Change runtime type** -> chọn **T4 GPU**.
2. **Chạy Cell 1**: Cài đặt các thư viện cần thiết.
3. **Chạy Cell 2**: (Tùy chọn nhập NGROK Authtoken) Khởi động máy chủ và dán đường dẫn nhận được vào tab Cài Đặt của ứng dụng.

In [ ]:
#@title Bước 1: Cài đặt các thư viện cần thiết
!pip install flask vieneu torch torchaudio pyngrok

In [ ]:
# ==============================================================================
# VIGEN AIO STUDIO - GOOGLE COLAB SERVER FOR VIENEU-TTS (ASYNC, NGROK & GDRIVE SUPPORT)
# ==============================================================================
#@title Bước 2: Khởi chạy Máy chủ VieNeu-TTS & Đường truyền (Ngrok / Cloudflare)
NGROK_AUTHTOKEN = "" #@param {type:"string"}

import os
import re
import time
import uuid
import tempfile
import threading
import subprocess
import torch
import torchaudio
from flask import Flask, request, send_file, jsonify

app = Flask(__name__)
TEMP_DIR = tempfile.gettempdir()
tasks = {}

def cleanup_old_tasks():
    now = time.time()
    for tid in list(tasks.keys()):
        if now - tasks[tid].get("created_at", 0) > 3600:
            task_info = tasks.pop(tid, None)
            if task_info and task_info.get("output_path") and os.path.exists(task_info["output_path"]):
                try: os.remove(task_info["output_path"])
                except Exception: pass

print("Đang khởi tạo mô hình VieNeu-TTS...")
try:
    from vieneu import VieNeuTTS
    tts = VieNeuTTS()
    print("Khởi tạo mô hình VieNeu-TTS thành công!")
except Exception as e:
    print(f"LỖI khởi tạo mô hình VieNeu-TTS: {str(e)}")

@app.route('/upload_ref', methods=['POST'])
def upload_ref():
    if 'file' not in request.files: return jsonify({"error": "No file"}), 400
    file = request.files['file']
    file_path = os.path.join(TEMP_DIR, f"vieneu_ref_{os.urandom(4).hex()}.wav")
    file.save(file_path)
    return jsonify({"ref_path": file_path})

@app.route('/synthesize_async', methods=['POST'])
def synthesize_async():
    cleanup_old_tasks()
    text = request.form.get('text', '')
    voice = request.form.get('voice', '')
    ref_path = request.form.get('ref_path', '')
    if not text: return jsonify({"error": "Text required"}), 400
    
    ref_audio_file = None
    if ref_path and os.path.exists(ref_path):
        ref_audio_file = ref_path
    elif 'file' in request.files:
        file = request.files['file']
        if file.filename != '':
            ref_audio_file = os.path.join(TEMP_DIR, f"vieneu_ref_{os.urandom(4).hex()}.wav")
            file.save(ref_audio_file)

    task_id = str(uuid.uuid4())
    tasks[task_id] = {"status": "processing", "output_path": None, "error": None, "created_at": time.time()}

    def run_synthesis_job(tid, txt, v_name, r_file):
        output_path = os.path.join(TEMP_DIR, f"vieneu_out_{tid}.wav")
        try:
            if r_file: audio = tts.infer(text=txt, ref_audio=r_file)
            elif v_name: audio = tts.infer(text=txt, voice=v_name)
            else: audio = tts.infer(text=txt)
            tts.save(audio, output_path)
            
            drive_info = {}
            if os.path.exists("/content/drive/MyDrive"):
                try:
                    gdrive_dir = "/content/drive/MyDrive/ViGen_AIO_Audio"
                    os.makedirs(gdrive_dir, exist_ok=True)
                    gdrive_path = os.path.join(gdrive_dir, f"vieneu_out_{tid}.wav")
                    import shutil
                    shutil.copy2(output_path, gdrive_path)
                    drive_info["gdrive_path"] = gdrive_path
                    try:
                        from googleapiclient.discovery import build
                        from google.colab import auth
                        auth.authenticate_user()
                        drive_service = build('drive', 'v3')
                        filename = f"vieneu_out_{tid}.wav"
                        results = drive_service.files().list(q=f"name='{filename}' and trashed=false", fields="files(id, name)").execute()
                        items = results.get('files', [])
                        if items:
                            file_id = items[0]['id']
                            drive_service.permissions().create(fileId=file_id, body={'type': 'anyone', 'role': 'reader'}).execute()
                            drive_info["drive_file_id"] = file_id
                    except Exception:
                        pass
                except Exception: pass

            tasks[tid]["status"] = "completed"
            tasks[tid]["output_path"] = output_path
            tasks[tid]["file_size"] = os.path.getsize(output_path) if os.path.exists(output_path) else 0
            tasks[tid]["drive_info"] = drive_info
        except Exception as e:
            tasks[tid]["status"] = "error"
            tasks[tid]["error"] = str(e)

    threading.Thread(target=run_synthesis_job, args=(task_id, text, voice, ref_audio_file), daemon=True).start()
    return jsonify({"task_id": task_id, "status": "processing"})

@app.route('/status/<task_id>', methods=['GET'])
def get_status(task_id):
    if task_id not in tasks: return jsonify({"error": "Not found"}), 404
    info = tasks[task_id]
    return jsonify({"task_id": task_id, "status": info["status"], "file_size": info.get("file_size", 0), "error": info.get("error"), "drive_info": info.get("drive_info", {})})

@app.route('/download/<task_id>', methods=['GET'])
def download_result(task_id):
    if task_id not in tasks: return jsonify({"error": "Not found"}), 404
    info = tasks[task_id]
    if info["status"] != "completed" or not info.get("output_path") or not os.path.exists(info["output_path"]):
        return jsonify({"error": "Not ready"}), 400
    return send_file(info["output_path"], mimetype="audio/wav")

@app.route('/delete/<task_id>', methods=['POST', 'DELETE', 'GET'])
def delete_task_file(task_id):
    info = tasks.pop(task_id, {})
    deleted = []
    if info.get("output_path") and os.path.exists(info["output_path"]):
        try: os.remove(info["output_path"]); deleted.append(info["output_path"])
        except Exception: pass
    drive_info = info.get("drive_info", {})
    if drive_info.get("gdrive_path") and os.path.exists(drive_info["gdrive_path"]):
        try: os.remove(drive_info["gdrive_path"]); deleted.append(drive_info["gdrive_path"])
        except Exception: pass
    return jsonify({"success": True, "deleted": deleted})

@app.route('/synthesize', methods=['POST'])
def synthesize():
    text = request.form.get('text', '')
    voice = request.form.get('voice', '')
    ref_path = request.form.get('ref_path', '')
    if not text: return jsonify({"error": "Text required"}), 400
    ref_audio_file = ref_path if (ref_path and os.path.exists(ref_path)) else None
    if not ref_audio_file and 'file' in request.files:
        file = request.files['file']
        if file.filename != '':
            ref_audio_file = os.path.join(TEMP_DIR, f"vieneu_ref_{os.urandom(4).hex()}.wav")
            file.save(ref_audio_file)
    output_path = os.path.join(TEMP_DIR, f"vieneu_out_{os.urandom(4).hex()}.wav")
    if ref_audio_file: audio = tts.infer(text=text, ref_audio=ref_audio_file)
    elif voice: audio = tts.infer(text=text, voice=voice)
    else: audio = tts.infer(text=text)
    tts.save(audio, output_path)
    return send_file(output_path, mimetype="audio/wav")

def start_tunnel():
    authtoken = NGROK_AUTHTOKEN.strip()
    if authtoken:
        print("\n⚡ Phát hiện Ngrok Authtoken! Đang khởi động đường truyền Ngrok...")
        try:
            subprocess.run(["pip", "install", "-q", "pyngrok"])
            from pyngrok import ngrok
            ngrok.set_auth_token(authtoken)
            tunnel = ngrok.connect(39828)
            public_url = tunnel.public_url
            if public_url.startswith("http://"): public_url = public_url.replace("http://", "https://")
            print("\n" + "="*55 + f"\n 🚀 LIÊN KẾT ĐƯỜNG TRUYỀN NGROK CỦA BẠN (ƯU TIÊN):\n {public_url}\n" + "="*55)
            return
        except Exception as ne:
            print(f"⚠️ Ngrok lỗi: {ne}. Đang tự chuyển sang Cloudflare Tunnel...")

    print("\n🌐 Đang cài đặt và khởi tạo Cloudflare Tunnel...")
    subprocess.run(["wget", "-q", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb"])
    subprocess.run(["dpkg", "-i", "cloudflared-linux-amd64.deb"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    proc = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:39828"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for _ in range(30):
        time.sleep(1)
        line = proc.stdout.readline()
        if "trycloudflare.com" in line:
            match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
            if match:
                print("\n" + "="*55 + f"\n 🌐 LIÊN KẾT ĐƯỜNG TRUYỀN CLOUDFLARE CỦA BẠN:\n {match.group(0)}\n" + "="*55)
                break

if __name__ == '__main__':
    threading.Thread(target=start_tunnel, daemon=True).start()
    print("Đang khởi chạy Flask server trên cổng 39828...")
    app.run(port=39828, host='0.0.0.0')
